# Neuro101DD — Data Analysis Tutorial

## Part 3: alignment, counting and statistics

Work through this after you have finished Part 2 of the worksheet. Everything here depends on the filter and threshold you chose there, so have those numbers to hand before you start.


In [ ]:
#RUN THIS FIRST -- about a minute, once per session.
# Versions are pinned deliberately; see the note in the Part 1 notebook.
!pip -q install "spikeinterface==0.104.9" "probeinterface==0.4.0" "neo==0.14.5"
!pip -q install "tridesclous==1.6.9" pyarrow
print("install finished")


In [ ]:
#RUN THIS SECOND, every time you open or reset the session.
import sys, pathlib, subprocess
import numpy as np

# NumPy 2 compatibility -- see the Part 1 notebook for why this is here.
for _old, _new in [("in1d", "isin"), ("alltrue", "all"), ("sometrue", "any"),
                   ("row_stack", "vstack"), ("trapz", "trapezoid")]:
    if not hasattr(np, _old) and hasattr(np, _new):
        setattr(np, _old, getattr(np, _new))

BASE = "https://raw.githubusercontent.com/mhburrell/Neuro101DD_Tutorial/refs/heads/main/"
for src, dst in [("n101dd_functions.py", "workshop_utils.py"),
                 ("n101dd_analysis.py", "n101dd_analysis.py")]:
    subprocess.check_call(["curl", "-sSL", BASE + src, "-o", dst])
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from workshop_utils import run_sorting, plot_trial, plot_all_trials, filter_recording
from n101dd_analysis import (
    build_sorted_spikes, load_sorted_spikes, recording_summary,
    plot_raster_psth_all, plot_raster_psth_by_volume,
    count_spikes_by_volume, response_table,
    cue_response_vs_volume, outcome_response_vs_volume,
    outcome_response_vs_error, same_reward_different_cue,
    stats_before_after,
)
print("setup ok")


In [ ]:
#RUN THIS THIRD -- downloads YOUR ten recordings (about 160 MB, a minute or so).
# Put the code you were emailed between the quotes. It is not the same as anyone
# else's, and neither are your recordings.

STUDENT_CODE = ""          # <-- your code goes here

import pathlib, subprocess, zipfile

def _in_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True

if not _in_colab():
    raise RuntimeError(
        "This notebook fetches your data only when it is running in Google Colab, and "
        "this is not Colab.\n\n"
        "Open the notebook at https://colab.research.google.com and run it there.\n\n"
        "If you ARE in Colab and seeing this message, something has changed at Google's "
        "end -- email me and I will fix it. Please do not edit this cell to get around it, "
        "because the data you would be working on then is not yours."
    )

assert STUDENT_CODE, "Put your student code in STUDENT_CODE above before running this."
URL = f"https://github.com/mhburrell/Neuro101DD_Tutorial/releases/download/Data2026/{STUDENT_CODE}.zip"
dest = pathlib.Path("/content/data")
if not any(dest.glob("recording*_events.json")):
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["curl", "-sSL", "--fail", URL, "-o", "/content/mydata.zip"])
    with zipfile.ZipFile("/content/mydata.zip") as z:
        z.extractall(dest)
    inner = [p for p in dest.iterdir() if p.is_dir() and (p / "dataset_id.txt").exists()]
    if inner:                                  # flatten a single wrapping folder
        for item in inner[0].iterdir():
            item.rename(dest / item.name)

print("recordings available:",
      sorted(int(p.name.replace("recording", "").replace("_events.json", ""))
             for p in dest.glob("recording*_events.json")))
stamp = dest / "dataset_id.txt"
if stamp.exists():
    print("\ndata set:", stamp.read_text().strip())
    print("Paste that line at the top of your answer to Q2.1.")


# Building your sorted spikes

In Part 2 you sorted recordings one at a time to look at them. Now you need the sorted spikes from all of them in one table, lined up with what happened on each trial — which cue appeared, how much juice was delivered, and when.

`build_sorted_spikes` does that, and it takes **your** filter settings and **your** detection threshold as arguments. Those values decide which spikes end up in the table, and so decide every number you report from here on.

If you change your mind about a parameter, run this cell again. Whatever values you end up using belong in your methods paragraph (Q3.7).


In [ ]:
# Put in the values YOU settled on in Part 2.
# Ten recordings takes a couple of minutes.

spikes = build_sorted_spikes(detect_threshold=5, freq_min=300, freq_max=6000)


## A first look at what you have

One row per unit, with its spike count, its firing rate and how many trials it appears in. Read it against the identification criteria you listed in Q1.5, and against what you saw in Part 2 — the waveform is not in this table.

If two units in one recording cover different sets of trials, ask yourself what the sorter was actually doing.


In [ ]:
print(recording_summary(spikes).to_string(index=False))


# Rasters and PSTHs

## plot_raster_psth_all

Every trial of one cell, in the order it was recorded, aligned either to the cue or to the outcome. Because the trials are in recording order, anything that changed partway through the session appears as a change down the page.

`align_to='outcome'` lines up the moment the juice arrived — or would have arrived, on trials where it did not.


In [ ]:
plot_raster_psth_all(recording_num=1, cell_num=1, align_to='cue',
                     t_window_ms=(-500, 1500), bin_size_ms=50)


## plot_raster_psth_by_volume

The same thing, but one panel per volume so the panels can be compared. With `align_to='cue'` the panels are the volume the cue predicted; with `align_to='outcome'` they are the volume delivered.

`matched_only=True` (the default) keeps only the fully predicted trials, which is what you want for Q3.3 and Q3.4. Set it to `False` to bring the probe trials in.


In [ ]:
plot_raster_psth_by_volume(recording_num=1, cell_num=1, align_to='cue',
                           t_window_ms=(-500, 1500), bin_size_ms=50, matched_only=True)


# Counting spikes

A picture is not a measurement. To compare conditions you need a number per trial: the count of spikes in a window after the event. Choosing that window is Q3.2, and you should use the same window for the rest of Part 3.

Look at the min and max columns as well as the mean.


In [ ]:
count_spikes_by_volume(recording_num=1, cell_num=1, align_to='cue', window_ms=(50, 300))


# Is there a response at all?

Before asking whether a response varies with volume, check that there is a response. `stats_before_after` compares spike counts in a window before the event with a window after it, trial by trial.

The default test is the Wilcoxon rank-sum rather than a t-test; Q3.7 should say which you used and why. `subset` picks the trials: `'matched'`, `'probe'`, `'uncued'` or `'all'`.


In [ ]:
stats_before_after(recording_num=1, cell_num=1, align_to='outcome',
                   before_ms=(-350, -50), after_ms=(50, 350), subset='uncued')


# The two regressions

Each of these fits a straight line through the single-trial counts, reports a slope, an r² and a p value, and draws the scatter with the fit over it so you can see whether a straight line was a reasonable thing to fit.

Both use all five volumes in one test rather than comparing volumes pairwise. Part of Q3.7 is saying why that is the analysis you ran.

## Q3.3 — the cue response against the volume predicted


In [ ]:
cue_response_vs_volume(recording_num=1, cell_num=1, window_ms=(50, 300))


## Q3.4 — the outcome response against the volume delivered

Fully predicted trials only: on every trial in this test, the animal got exactly what the cue predicted.

**Write down what you expect this slope to be before you run the cell**, and make sure you can say what each of the two accounts in Q1.1 predicts for it. Then answer Q3.4 against what you predicted rather than against what comes out.


In [ ]:
outcome_response_vs_volume(recording_num=1, cell_num=1, window_ms=(50, 350))


## The outcome response against the prediction error

Every cued trial now, plotted against `delta` — the difference between what arrived and what was predicted. Negative where the animal got less than the cue predicted.


In [ ]:
outcome_response_vs_error(recording_num=1, cell_num=1, window_ms=(50, 350))


# Q3.5 — the same reward, differently predicted

The mismatch trials hold the juice fixed and change what was expected. 0.25 ml and 0.40 ml each arrive after a smaller cue, after their own cue, and after a larger cue.


In [ ]:
same_reward_different_cue(recording_num=1, cell_num=1, window_ms=(50, 350))


# Everything for one cell in one table

Cue and outcome response for every combination of predicted and delivered volume, with the number of trials behind each one. Useful when you come to write up, and a good way to spot a condition you have been reading too much into when it only had a handful of trials.


In [ ]:
response_table(recording_num=1, cell_num=1,
               cue_window_ms=(50, 300), outcome_window_ms=(50, 350))


---

**Before you write up.** Q3.7 asks for methods somebody else could reproduce your numbers from. That means the filter band and detection threshold you passed to `build_sorted_spikes`, which recordings you kept and which you left out and why, your two time windows, and which tests you ran. If you had to rebuild the table partway through because you changed a parameter, the values in your write-up should be the ones your final numbers came from.
